In [ ]:
# import os
# import pandas as pd
# import pyarrow.parquet as pq

# def carica_universo_estremo_fallback(cartella_sorgente="progetto_trading_data"):
#     if not os.path.exists(cartella_sorgente):
#         print(f"❌ Errore: La cartella '{cartella_sorgente}' non esiste.")
#         return None
        
#     dati_ricaricati = {}
#     print("🔄 Caricamento via PyArrow Table -> Pure Python Dict -> Pandas Constructor...")
    
#     for file_name in os.listdir(cartella_sorgente):
#         # Escludiamo eventuali file di configurazione come il JSON della mappa
#         if file_name.endswith(".parquet"):
#             cluster = file_name.replace(".parquet", "")
#             dati_ricaricati[cluster] = {}
#             file_path = os.path.join(cartella_sorgente, file_name)
            
#             # 1. Leggiamo il file come tabella nativa PyArrow
#             tabella_pyarrow = pq.read_table(file_path)
            
#             # 2. TRICK: Convertiamo in un dizionario standard di Python (liste pure)
#             # Questo evita l'uso di .to_pandas() e distrugge ogni conflitto di librerie
#             dati_dizionario = tabella_pyarrow.to_pydict()
            
#             # 3. Alleviamo il DataFrame passando dal costruttore standard di Pandas
#             df_cluster = pd.DataFrame(dati_dizionario)
            
#             # 4. Smistiamo i ticker nel dizionario per il backtest
#             for ticker, group in df_cluster.groupby('Ticker'):
#                 df_ticker = group.drop(columns=['Ticker']).set_index('time')
#                 df_ticker.index = pd.to_datetime(df_ticker.index)
#                 dati_ricaricati[cluster][ticker] = df_ticker
                
#     tot_ticker = sum(len(t) for t in dati_ricaricati.values())
#     print(f"\n✅ Vittoria! {tot_ticker} ticker pronti in 'dati_puliti' senza alcun conflitto di versione.")
#     return dati_ricaricati

# # Inizializza finalmente la variabile RAM
# dati_puliti = carica_universo_estremo_fallback()

🔄 Caricamento via PyArrow Table -> Pure Python Dict -> Pandas Constructor...

✅ Vittoria! 124 ticker pronti in 'dati_puliti' senza alcun conflitto di versione.


In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# from scipy.ndimage import generic_filter

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 (Filtro Altopiano Robusto Anti-Overfit) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: COSTRUTTORE SPLIT 60-40 (CONV 3D ROBUSTA + ALPHA)
# # ==============================================================================

# # 🔥 IL NUOVO FILTRO ANTI-OVERFIT: Media - Penalità di Varianza
# def robust_plateau_filter(buffer):
#     valid_cells = buffer[~np.isnan(buffer)]
#     # Esigiamo almeno 9 vicini validi (su 27 celle possibili nel cubo 3x3x3)
#     if len(valid_cells) < 9:
#         return np.nan
    
#     media = np.mean(valid_cells)
#     dev_std = np.std(valid_cells)
    
#     # Lo score dell'altopiano è la media decurtata della sua instabilità (rugosità)
#     # Se ci sono picchi anomali, dev_std esplode e distrugge lo score.
#     score_altopiano = media - dev_std
#     return score_altopiano

# def ottimizza_asset_split_60_40(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
    
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_cols = [t[0] for t in griglia]
#     m_cols = [t[1] for t in griglia]
#     s_cols = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
#     # --- FASE 1: IN-SAMPLE (60% TRAIN) ---
#     close_is = close.iloc[:split_idx].reset_index(drop=True)
#     ema_is = ema_df.iloc[:split_idx].reset_index(drop=True)
    
#     ema1_mat = ema_is[f_cols]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_is[m_cols]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_is[s_cols]; ema3_mat.columns = multi_idx
    
#     entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
#     exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
#     entries_is = entries_raw.vbt.signals.fshift(1)
#     exits_is = exits_raw.vbt.signals.fshift(1)
    
#     pf_is = vbt.Portfolio.from_signals(
#         close_is, entries_is, exits_is, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     rets_is = pf_is.returns()
#     std_is = rets_is.std()
    
#     anni_totali_is = len(close_is) / giorni_anno
#     trades_is_array = pf_is.trades.count().reindex(multi_idx, fill_value=0).values
#     tpy_is_array = trades_is_array / anni_totali_is if anni_totali_is > 0 else 0
    
#     sharpe_array_is = np.where(
#         (std_is > 0) & (tpy_is_array >= 2), 
#         (rets_is.mean() / std_is) * np.sqrt(giorni_anno), 
#         np.nan
#     )
    
#     # --- FASE 2: CONVOLUZIONE 3D CON PENALITA' DI VARIANZA ---
#     shape_3d = (len(fast_range), len(med_range), len(slow_range))
#     cubo_sharpe = np.full(shape_3d, np.nan)
    
#     for idx, (f, m, s) in enumerate(griglia):
#         fi = np.where(fast_range == f)[0][0]
#         mi = np.where(med_range == m)[0][0]
#         si = np.where(slow_range == s)[0][0]
#         cubo_sharpe[fi, mi, si] = sharpe_array_is[idx]
        
#     # Applichiamo il filtro robusto che penalizza gli spuntoni (overfitting)
#     cubo_stabilizzato = generic_filter(cubo_sharpe, robust_plateau_filter, size=3, mode='constant', cval=np.nan)
    
#     smoothed_valid_sharpes = np.full(len(griglia), np.nan)
#     for idx, (f, m, s) in enumerate(griglia):
#         if not np.isnan(sharpe_array_is[idx]):
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             smoothed_valid_sharpes[idx] = cubo_stabilizzato[fi, mi, si]
        
#     if np.isnan(smoothed_valid_sharpes).all():
#         return "LOW_SHARPE"
        
#     best_idx = np.nanargmax(smoothed_valid_sharpes)
#     best_f, best_m, best_s = griglia[best_idx]
#     sharpe_is_scelto = sharpe_array_is[best_idx]
    
#     data_inizio_oos = close.index[split_idx].strftime('%Y-%m-%d')
    
#     # --- FASE 3: OUT-OF-SAMPLE (40% TEST) ---
#     date_oos = close.iloc[split_idx:].index
#     close_oos = close.iloc[split_idx:].reset_index(drop=True)
    
#     ema1_oos = ema_df[best_f].iloc[split_idx:].reset_index(drop=True)
#     ema2_oos = ema_df[best_m].iloc[split_idx:].reset_index(drop=True)
#     ema3_oos = ema_df[best_s].iloc[split_idx:].reset_index(drop=True)
    
#     entries_oos_raw = (ema1_oos.vbt.crossed_above(ema2_oos) | ema1_oos.vbt.crossed_above(ema3_oos) | ema2_oos.vbt.crossed_above(ema3_oos))
#     exits_oos_raw = (ema1_oos.vbt.crossed_below(ema2_oos) | ema1_oos.vbt.crossed_below(ema3_oos) | ema2_oos.vbt.crossed_below(ema3_oos))
    
#     entries_oos = entries_oos_raw.vbt.signals.fshift(1)
#     exits_oos = exits_oos_raw.vbt.signals.fshift(1)
    
#     pf_oos = vbt.Portfolio.from_signals(
#         close_oos, entries_oos, exits_oos, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     # --- FASE 4: METRICHE STRATEGIA VS BUY & HOLD ---
#     equity_returns_oos = pf_oos.returns()
#     equity_returns_oos.index = date_oos
#     equity_curve_oos = (1 + equity_returns_oos).cumprod()
    
#     giorni_totali_oos = len(equity_curve_oos)
#     anni_totali_oos = giorni_totali_oos / giorni_anno
    
#     cagr_strat = (equity_curve_oos.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if anni_totali_oos > 0 else 0
#     vol_strat = equity_returns_oos.std() * np.sqrt(giorni_anno)
#     sharpe_strat = (equity_returns_oos.mean() / equity_returns_oos.std() * np.sqrt(giorni_anno)) if vol_strat > 0 else 0
    
#     peak_strat = equity_curve_oos.cummax()
#     max_dd_strat = ((equity_curve_oos - peak_strat) / peak_strat).min()
#     total_trades_oos = pf_oos.trades.count()
    
#     close_oos_bh = close.loc[date_oos]
#     rets_bh = close_oos_bh.pct_change().dropna()
#     equity_bh = (1 + rets_bh).cumprod()
    
#     cagr_bh = (equity_bh.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_bh) > 0 and anni_totali_oos > 0 else 0
#     vol_bh = rets_bh.std() * np.sqrt(giorni_anno)
#     sharpe_bh = (rets_bh.mean() / rets_bh.std() * np.sqrt(giorni_anno)) if vol_bh > 0 else 0
    
#     peak_bh = equity_bh.cummax()
#     max_dd_bh = ((equity_bh - peak_bh) / peak_bh).min() if len(equity_bh) > 0 else 0
    
#     alpha = cagr_strat - cagr_bh
    
#     return {
#         "Param (F-M-S)": f"{best_f}-{best_m}-{best_s}",
#         "Start_OOS": data_inizio_oos,
#         "Shp_IS": sharpe_is_scelto,
#         "Shp_OOS": sharpe_strat,
#         "Shp_B&H": sharpe_bh,
#         "CAGR_Str": cagr_strat * 100,
#         "CAGR_B&H": cagr_bh * 100,
#         "Alpha_%": alpha * 100,
#         "DD_Str": max_dd_strat * 100,
#         "DD_B&H": max_dd_bh * 100,
#         "TRD_OOS": total_trades_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"⌛ Split 60-40 per {ticker:<10} | Cluster: {cluster_appartenenza:<12} ... ", end="")
    
#     metrics = ottimizza_asset_split_60_40(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#         print(f"COMPLETATO ✅ (EMA: {metrics['Param (F-M-S)']})")
#     elif metrics == "LOW_SHARPE":
#         print("SCARTATO (Nessun altopiano valido trovato in IS) ⚠️")
#     else:
#         print("ERRORE/STORICO INSUFFICIENTE ❌")

# # ==============================================================================
# # 5. LEADERBOARD E STAMPA TABELLONE FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_%", ascending=False).reset_index(drop=True)

#     print("\n" + "="*145)
#     print("🏆 TABELLONE FINALE SPLIT 60/40 - FILTRO ANTI-OVERFIT (Ordinato per Alpha OOS vs B&H)")
#     print("="*145)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Start_OOS", "Param (F-M-S)", "Shp_IS", "Shp_OOS", "Shp_B&H", "CAGR_Str", "CAGR_B&H", "Alpha_%", "DD_Str", "DD_B&H", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "F-M-S", "SHP IS", "SHP OOS", "SHP B&H", "CAGR STR", "CAGR B&H", "ALPHA %", "DD STR", "DD B&H", "TRD"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'SHP B&H':   '{:,.2f}'.format,
#             'CAGR STR':  '{:,.1f}%'.format,
#             'CAGR B&H':  '{:,.1f}%'.format,
#             'ALPHA %':   '{:,.1f}%'.format,
#             'DD STR':    '{:,.1f}%'.format,
#             'DD B&H':    '{:,.1f}%'.format,
#             'TRD':       '{:,}'.format
#         }
#     ))
#     print("="*145)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 (Filtro Altopiano Robusto Anti-Overfit) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.
⌛ Split 60-40 per US100.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 4-60-155)
⌛ Split 60-40 per US30.cash  | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 22-64-160)
⌛ Split 60-40 per US500.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 10-28-235)
⌛ Split 60-40 per BNBUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-180)
⌛ Split 60-40 per BTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 8-40-155)
⌛ Split 60-40 per ETHUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-235)
⌛ Split 60-40 per GRTUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 18-36-170)
⌛ Split 60-40 per LTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 16-28-215)
⌛ Split 60-40 per MANUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-200)
⌛ Split 60-40 per NERUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 16-20-100

# IN SAMPLE SPLIT per robustezza

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# from scipy.ndimage import generic_filter

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 (Conv 3D + IS Sub-Period Cross-Validation) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: COSTRUTTORE SPLIT 60-40 CON IS SUB-PERIOD ANALYSIS
# # ==============================================================================
# def nanmean_filter(buffer):
#     valid_count = np.sum(~np.isnan(buffer))
#     if valid_count < 9:
#         return np.nan
#     return np.nanmean(buffer)

# def ottimizza_asset_split_60_40(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
    
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_cols = [t[0] for t in griglia]
#     m_cols = [t[1] for t in griglia]
#     s_cols = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
#     # --- FASE 1: IN-SAMPLE (60% TRAIN) ---
#     close_is = close.iloc[:split_idx].reset_index(drop=True)
#     ema_is = ema_df.iloc[:split_idx].reset_index(drop=True)
    
#     ema1_mat = ema_is[f_cols]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_is[m_cols]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_is[s_cols]; ema3_mat.columns = multi_idx
    
#     entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
#     exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
#     entries_is = entries_raw.vbt.signals.fshift(1)
#     exits_is = exits_raw.vbt.signals.fshift(1)
    
#     pf_is = vbt.Portfolio.from_signals(
#         close_is, entries_is, exits_is, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     rets_is = pf_is.returns()
    
#     # CALCOLO DELLO SHARPE GLOBALE IN-SAMPLE (solo per reporting)
#     std_is_tot = rets_is.std()
#     sharpe_array_is_tot = np.where(std_is_tot > 0, (rets_is.mean() / std_is_tot) * np.sqrt(giorni_anno), np.nan)
    
#     # 🔥 FASE 1.5: SUB-PERIOD ANALYSIS (Tagliamo l'IS a metà) 🔥
#     half_idx = len(rets_is) // 2
#     rets_is_1 = rets_is.iloc[:half_idx]
#     rets_is_2 = rets_is.iloc[half_idx:]
    
#     std_1 = rets_is_1.std()
#     shp_1 = np.where(std_1 > 0, (rets_is_1.mean() / std_1) * np.sqrt(giorni_anno), -99.0)
    
#     std_2 = rets_is_2.std()
#     shp_2 = np.where(std_2 > 0, (rets_is_2.mean() / std_2) * np.sqrt(giorni_anno), -99.0)
    
#     anni_totali_is = len(close_is) / giorni_anno
#     trades_is_array = pf_is.trades.count().reindex(multi_idx, fill_value=0).values
#     tpy_is_array = trades_is_array / anni_totali_is if anni_totali_is > 0 else 0
    
#     # SCORE DI COERENZA IS: Richiede profitti in ENTRAMBE le metà dell'IS
#     # Penalizza le combo che hanno enorme differenza di performance tra i due periodi
#     score_coerenza_is = np.where(
#         (shp_1 > 0) & (shp_2 > 0) & (tpy_is_array >= 2),
#         ((shp_1 + shp_2) / 2) - np.abs(shp_1 - shp_2),
#         np.nan
#     )
    
#     # --- FASE 2: CONVOLUZIONE 3D SULLO SCORE DI COERENZA ---
#     shape_3d = (len(fast_range), len(med_range), len(slow_range))
#     cubo_coerenza = np.full(shape_3d, np.nan)
    
#     for idx, (f, m, s) in enumerate(griglia):
#         fi = np.where(fast_range == f)[0][0]
#         mi = np.where(med_range == m)[0][0]
#         si = np.where(slow_range == s)[0][0]
#         cubo_coerenza[fi, mi, si] = score_coerenza_is[idx]
        
#     cubo_stabilizzato = generic_filter(cubo_coerenza, nanmean_filter, size=3, mode='constant', cval=np.nan)
    
#     smoothed_valid_scores = np.full(len(griglia), np.nan)
#     for idx, (f, m, s) in enumerate(griglia):
#         # La cella deve essere valida nel mondo reale (non NaN) per essere accettata
#         if not np.isnan(score_coerenza_is[idx]):
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             smoothed_valid_scores[idx] = cubo_stabilizzato[fi, mi, si]
        
#     if np.isnan(smoothed_valid_scores).all():
#         return "LOW_SHARPE"
        
#     best_idx = np.nanargmax(smoothed_valid_scores)
#     best_f, best_m, best_s = griglia[best_idx]
    
#     # Salviamo lo Sharpe Globale IS della combo scelta per metterlo a tabellone
#     sharpe_is_scelto = sharpe_array_is_tot[best_idx]
    
#     data_inizio_oos = close.index[split_idx].strftime('%Y-%m-%d')
    
#     # --- FASE 3: OUT-OF-SAMPLE (40% TEST) ---
#     date_oos = close.iloc[split_idx:].index
#     close_oos = close.iloc[split_idx:].reset_index(drop=True)
    
#     ema1_oos = ema_df[best_f].iloc[split_idx:].reset_index(drop=True)
#     ema2_oos = ema_df[best_m].iloc[split_idx:].reset_index(drop=True)
#     ema3_oos = ema_df[best_s].iloc[split_idx:].reset_index(drop=True)
    
#     entries_oos_raw = (ema1_oos.vbt.crossed_above(ema2_oos) | ema1_oos.vbt.crossed_above(ema3_oos) | ema2_oos.vbt.crossed_above(ema3_oos))
#     exits_oos_raw = (ema1_oos.vbt.crossed_below(ema2_oos) | ema1_oos.vbt.crossed_below(ema3_oos) | ema2_oos.vbt.crossed_below(ema3_oos))
    
#     entries_oos = entries_oos_raw.vbt.signals.fshift(1)
#     exits_oos = exits_oos_raw.vbt.signals.fshift(1)
    
#     pf_oos = vbt.Portfolio.from_signals(
#         close_oos, entries_oos, exits_oos, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     # --- FASE 4: METRICHE STRATEGIA VS BUY & HOLD ---
#     equity_returns_oos = pf_oos.returns()
#     equity_returns_oos.index = date_oos
#     equity_curve_oos = (1 + equity_returns_oos).cumprod()
    
#     giorni_totali_oos = len(equity_curve_oos)
#     anni_totali_oos = giorni_totali_oos / giorni_anno
    
#     cagr_strat = (equity_curve_oos.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if anni_totali_oos > 0 else 0
#     vol_strat = equity_returns_oos.std() * np.sqrt(giorni_anno)
#     sharpe_strat = (equity_returns_oos.mean() / equity_returns_oos.std() * np.sqrt(giorni_anno)) if vol_strat > 0 else 0
    
#     peak_strat = equity_curve_oos.cummax()
#     max_dd_strat = ((equity_curve_oos - peak_strat) / peak_strat).min() if len(equity_curve_oos) > 0 else 0
#     total_trades_oos = pf_oos.trades.count()
    
#     close_oos_bh = close.loc[date_oos]
#     rets_bh = close_oos_bh.pct_change().dropna()
#     equity_bh = (1 + rets_bh).cumprod()
    
#     cagr_bh = (equity_bh.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_bh) > 0 and anni_totali_oos > 0 else 0
#     vol_bh = rets_bh.std() * np.sqrt(giorni_anno)
#     sharpe_bh = (rets_bh.mean() / rets_bh.std() * np.sqrt(giorni_anno)) if vol_bh > 0 else 0
    
#     peak_bh = equity_bh.cummax()
#     max_dd_bh = ((equity_bh - peak_bh) / peak_bh).min() if len(equity_bh) > 0 else 0
    
#     alpha = cagr_strat - cagr_bh
    
#     return {
#         "Param (F-M-S)": f"{best_f}-{best_m}-{best_s}",
#         "Start_OOS": data_inizio_oos,
#         "Shp_IS": sharpe_is_scelto,
#         "Shp_OOS": sharpe_strat,
#         "Shp_B&H": sharpe_bh,
#         "CAGR_Str": cagr_strat * 100,
#         "CAGR_B&H": cagr_bh * 100,
#         "Alpha_%": alpha * 100,
#         "DD_Str": max_dd_strat * 100,
#         "DD_B&H": max_dd_bh * 100,
#         "TRD_OOS": total_trades_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"⌛ Split 60-40 per {ticker:<10} | Cluster: {cluster_appartenenza:<12} ... ", end="")
    
#     metrics = ottimizza_asset_split_60_40(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#         print(f"COMPLETATO ✅ (EMA: {metrics['Param (F-M-S)']})")
#     elif metrics == "LOW_SHARPE":
#         print("SCARTATO (Nessuna combo coerente nei due semi-periodi IS) ⚠️")
#     else:
#         print("ERRORE/STORICO INSUFFICIENTE ❌")

# # ==============================================================================
# # 5. LEADERBOARD E STAMPA TABELLONE FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_%", ascending=False).reset_index(drop=True)

#     print("\n" + "="*145)
#     print("🏆 TABELLONE FINALE SPLIT 60/40 - IS SUB-PERIOD CROSS VALIDATION (Ordinato per Alpha OOS)")
#     print("="*145)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Start_OOS", "Param (F-M-S)", "Shp_IS", "Shp_OOS", "Shp_B&H", "CAGR_Str", "CAGR_B&H", "Alpha_%", "DD_Str", "DD_B&H", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "F-M-S", "SHP IS", "SHP OOS", "SHP B&H", "CAGR STR", "CAGR B&H", "ALPHA %", "DD STR", "DD B&H", "TRD"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'SHP B&H':   '{:,.2f}'.format,
#             'CAGR STR':  '{:,.1f}%'.format,
#             'CAGR B&H':  '{:,.1f}%'.format,
#             'ALPHA %':   '{:,.1f}%'.format,
#             'DD STR':    '{:,.1f}%'.format,
#             'DD B&H':    '{:,.1f}%'.format,
#             'TRD':       '{:,}'.format
#         }
#     ))
#     print("="*145)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 (Conv 3D + IS Sub-Period Cross-Validation) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.
⌛ Split 60-40 per US100.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 4-64-230)
⌛ Split 60-40 per US30.cash  | Cluster: cash_cfd     ... SCARTATO (Nessuna combo coerente nei due semi-periodi IS) ⚠️
⌛ Split 60-40 per US500.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 12-32-205)
⌛ Split 60-40 per BNBUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 8-20-170)
⌛ Split 60-40 per BTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 24-56-195)
⌛ Split 60-40 per ETHUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 24-60-180)
⌛ Split 60-40 per GRTUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-36-245)
⌛ Split 60-40 per LTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 34-92-165)
⌛ Split 60-40 per MANUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-48-220)
⌛ Split 60-40 per NERUSD     | Cluster: crypto_cfd

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# from scipy.ndimage import generic_filter

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 (Filtro PESSIMAX: Il Peggior Vicino) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: COSTRUTTORE SPLIT 60-40 CON FILTRO PESSIMAX
# # ==============================================================================

# # 🔥 IL NUOVO FILTRO PESSIMAX: Il punteggio della cella è uguale al suo peggior vicino
# def pessimax_filter(buffer):
#     valid_cells = buffer[~np.isnan(buffer)]
#     # Esigiamo almeno 12 vicini validi (su 27). Vogliamo un altopiano molto denso.
#     if len(valid_cells) < 12:
#         return np.nan
    
#     # La magia matematica: valutiamo la stabilità in base al worst-case scenario locale
#     score_altopiano = np.min(valid_cells)
#     return score_altopiano

# def ottimizza_asset_split_60_40(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
    
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_cols = [t[0] for t in griglia]
#     m_cols = [t[1] for t in griglia]
#     s_cols = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
#     # --- FASE 1: IN-SAMPLE (60% TRAIN) ---
#     close_is = close.iloc[:split_idx].reset_index(drop=True)
#     ema_is = ema_df.iloc[:split_idx].reset_index(drop=True)
    
#     ema1_mat = ema_is[f_cols]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_is[m_cols]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_is[s_cols]; ema3_mat.columns = multi_idx
    
#     entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
#     exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
#     entries_is = entries_raw.vbt.signals.fshift(1)
#     exits_is = exits_raw.vbt.signals.fshift(1)
    
#     pf_is = vbt.Portfolio.from_signals(
#         close_is, entries_is, exits_is, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     rets_is = pf_is.returns()
#     std_is = rets_is.std()
    
#     anni_totali_is = len(close_is) / giorni_anno
#     trades_is_array = pf_is.trades.count().reindex(multi_idx, fill_value=0).values
#     tpy_is_array = trades_is_array / anni_totali_is if anni_totali_is > 0 else 0
    
#     # Sharpe Puro IS. Nessun moltiplicatore.
#     sharpe_array_is = np.where(
#         (std_is > 0) & (tpy_is_array >= 2), 
#         (rets_is.mean() / std_is) * np.sqrt(giorni_anno), 
#         np.nan
#     )
    
#     # --- FASE 2: CONVOLUZIONE 3D CON FILTRO PESSIMAX ---
#     shape_3d = (len(fast_range), len(med_range), len(slow_range))
#     cubo_sharpe = np.full(shape_3d, np.nan)
    
#     for idx, (f, m, s) in enumerate(griglia):
#         fi = np.where(fast_range == f)[0][0]
#         mi = np.where(med_range == m)[0][0]
#         si = np.where(slow_range == s)[0][0]
#         cubo_sharpe[fi, mi, si] = sharpe_array_is[idx]
        
#     cubo_stabilizzato = generic_filter(cubo_sharpe, pessimax_filter, size=3, mode='constant', cval=np.nan)
    
#     smoothed_valid_scores = np.full(len(griglia), np.nan)
#     for idx, (f, m, s) in enumerate(griglia):
#         if not np.isnan(sharpe_array_is[idx]):
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             smoothed_valid_scores[idx] = cubo_stabilizzato[fi, mi, si]
        
#     if np.isnan(smoothed_valid_scores).all():
#         return "LOW_SHARPE"
        
#     best_idx = np.nanargmax(smoothed_valid_scores)
#     best_f, best_m, best_s = griglia[best_idx]
    
#     sharpe_is_scelto = sharpe_array_is[best_idx]
#     data_inizio_oos = close.index[split_idx].strftime('%Y-%m-%d')
    
#     # --- FASE 3: OUT-OF-SAMPLE (40% TEST) ---
#     date_oos = close.iloc[split_idx:].index
#     close_oos = close.iloc[split_idx:].reset_index(drop=True)
    
#     ema1_oos = ema_df[best_f].iloc[split_idx:].reset_index(drop=True)
#     ema2_oos = ema_df[best_m].iloc[split_idx:].reset_index(drop=True)
#     ema3_oos = ema_df[best_s].iloc[split_idx:].reset_index(drop=True)
    
#     entries_oos_raw = (ema1_oos.vbt.crossed_above(ema2_oos) | ema1_oos.vbt.crossed_above(ema3_oos) | ema2_oos.vbt.crossed_above(ema3_oos))
#     exits_oos_raw = (ema1_oos.vbt.crossed_below(ema2_oos) | ema1_oos.vbt.crossed_below(ema3_oos) | ema2_oos.vbt.crossed_below(ema3_oos))
    
#     entries_oos = entries_oos_raw.vbt.signals.fshift(1)
#     exits_oos = exits_oos_raw.vbt.signals.fshift(1)
    
#     pf_oos = vbt.Portfolio.from_signals(
#         close_oos, entries_oos, exits_oos, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     # --- FASE 4: METRICHE STRATEGIA VS BUY & HOLD ---
#     equity_returns_oos = pf_oos.returns()
#     equity_returns_oos.index = date_oos
#     equity_curve_oos = (1 + equity_returns_oos).cumprod()
    
#     giorni_totali_oos = len(equity_curve_oos)
#     anni_totali_oos = giorni_totali_oos / giorni_anno
    
#     cagr_strat = (equity_curve_oos.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if anni_totali_oos > 0 else 0
#     vol_strat = equity_returns_oos.std() * np.sqrt(giorni_anno)
#     sharpe_strat = (equity_returns_oos.mean() / equity_returns_oos.std() * np.sqrt(giorni_anno)) if vol_strat > 0 else 0
    
#     peak_strat = equity_curve_oos.cummax()
#     max_dd_strat = ((equity_curve_oos - peak_strat) / peak_strat).min() if len(equity_curve_oos) > 0 else 0
#     total_trades_oos = pf_oos.trades.count()
    
#     close_oos_bh = close.loc[date_oos]
#     rets_bh = close_oos_bh.pct_change().dropna()
#     equity_bh = (1 + rets_bh).cumprod()
    
#     cagr_bh = (equity_bh.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_bh) > 0 and anni_totali_oos > 0 else 0
#     vol_bh = rets_bh.std() * np.sqrt(giorni_anno)
#     sharpe_bh = (rets_bh.mean() / rets_bh.std() * np.sqrt(giorni_anno)) if vol_bh > 0 else 0
    
#     peak_bh = equity_bh.cummax()
#     max_dd_bh = ((equity_bh - peak_bh) / peak_bh).min() if len(equity_bh) > 0 else 0
    
#     alpha = cagr_strat - cagr_bh
    
#     return {
#         "Param (F-M-S)": f"{best_f}-{best_m}-{best_s}",
#         "Start_OOS": data_inizio_oos,
#         "Shp_IS": sharpe_is_scelto,
#         "Shp_OOS": sharpe_strat,
#         "Shp_B&H": sharpe_bh,
#         "CAGR_Str": cagr_strat * 100,
#         "CAGR_B&H": cagr_bh * 100,
#         "Alpha_%": alpha * 100,
#         "DD_Str": max_dd_strat * 100,
#         "DD_B&H": max_dd_bh * 100,
#         "TRD_OOS": total_trades_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"⌛ Split 60-40 per {ticker:<10} | Cluster: {cluster_appartenenza:<12} ... ", end="")
    
#     metrics = ottimizza_asset_split_60_40(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#         print(f"COMPLETATO ✅ (EMA: {metrics['Param (F-M-S)']})")
#     elif metrics == "LOW_SHARPE":
#         print("SCARTATO (Nessun altopiano valido trovato in IS) ⚠️")
#     else:
#         print("ERRORE/STORICO INSUFFICIENTE ❌")

# # ==============================================================================
# # 5. LEADERBOARD E STAMPA TABELLONE FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_%", ascending=False).reset_index(drop=True)

#     print("\n" + "="*145)
#     print("🏆 TABELLONE FINALE SPLIT 60/40 - FILTRO PESSIMAX (Ordinato per Alpha OOS)")
#     print("="*145)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Start_OOS", "Param (F-M-S)", "Shp_IS", "Shp_OOS", "Shp_B&H", "CAGR_Str", "CAGR_B&H", "Alpha_%", "DD_Str", "DD_B&H", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "F-M-S", "SHP IS", "SHP OOS", "SHP B&H", "CAGR STR", "CAGR B&H", "ALPHA %", "DD STR", "DD B&H", "TRD"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'SHP B&H':   '{:,.2f}'.format,
#             'CAGR STR':  '{:,.1f}%'.format,
#             'CAGR B&H':  '{:,.1f}%'.format,
#             'ALPHA %':   '{:,.1f}%'.format,
#             'DD STR':    '{:,.1f}%'.format,
#             'DD B&H':    '{:,.1f}%'.format,
#             'TRD':       '{:,}'.format
#         }
#     ))
#     print("="*145)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 (Filtro PESSIMAX: Il Peggior Vicino) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.
⌛ Split 60-40 per US100.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 4-60-155)
⌛ Split 60-40 per US30.cash  | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 18-76-115)
⌛ Split 60-40 per US500.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 8-28-235)
⌛ Split 60-40 per BNBUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-185)
⌛ Split 60-40 per BTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 8-40-150)
⌛ Split 60-40 per ETHUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-230)
⌛ Split 60-40 per GRTUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 18-36-155)
⌛ Split 60-40 per LTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 16-28-215)
⌛ Split 60-40 per MANUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-200)
⌛ Split 60-40 per NERUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 16-20-100)
⌛ 

# grid + sens analysis

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 3EMA (Neighborhood Sensitivity Analysis) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: COSTRUTTORE SPLIT 60-40 CON SENSITIVITY ANALYSIS
# # ==============================================================================

# # Parametri per l'analisi di sensibilità
# TOLLERANZA_MAX_DEGRADO = 0.25 # Accettiamo massimo un calo del 25% nel vicinato
# MIN_VICINI_VALIDI = 6         # Richiediamo almeno N vicini validi per considerare la zona testabile

# def ottimizza_asset_split_60_40(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
    
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_cols = [t[0] for t in griglia]
#     m_cols = [t[1] for t in griglia]
#     s_cols = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
#     # --- FASE 1: IN-SAMPLE (60% TRAIN) ---
#     close_is = close.iloc[:split_idx].reset_index(drop=True)
#     ema_is = ema_df.iloc[:split_idx].reset_index(drop=True)
    
#     ema1_mat = ema_is[f_cols]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_is[m_cols]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_is[s_cols]; ema3_mat.columns = multi_idx
    
#     entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
#     exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
#     entries_is = entries_raw.vbt.signals.fshift(1)
#     exits_is = exits_raw.vbt.signals.fshift(1)
    
#     pf_is = vbt.Portfolio.from_signals(
#         close_is, entries_is, exits_is, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     rets_is = pf_is.returns()
#     std_is = rets_is.std()
    
#     anni_totali_is = len(close_is) / giorni_anno
#     trades_is_array = pf_is.trades.count().reindex(multi_idx, fill_value=0).values
#     tpy_is_array = trades_is_array / anni_totali_is if anni_totali_is > 0 else 0
    
#     # Calcolo Sharpe IS per tutte le combinazioni valide
#     sharpe_array_is = np.where(
#         (std_is > 0) & (tpy_is_array >= 2), 
#         (rets_is.mean() / std_is) * np.sqrt(giorni_anno), 
#         np.nan
#     )
    
#     # Calcolo Buy & Hold In-Sample
#     rets_bh_is = close_is.pct_change().dropna()
#     vol_bh_is = rets_bh_is.std() * np.sqrt(giorni_anno)
#     sharpe_bh_is = (rets_bh_is.mean() / rets_bh_is.std() * np.sqrt(giorni_anno)) if vol_bh_is > 0 else 0

#     # Se non ci sono strategie valide In-Sample, usciamo
#     if np.isnan(sharpe_array_is).all():
#         return "LOW_SHARPE"

#     # --- FASE 2: SENSITIVITY ANALYSIS (Ricerca del primo picco stabile) ---
    
#     # Creiamo una mappa tridimensionale per facilitare la ricerca dei vicini
#     shape_3d = (len(fast_range), len(med_range), len(slow_range))
#     cubo_sharpe = np.full(shape_3d, np.nan)
    
#     for idx, (f, m, s) in enumerate(griglia):
#         fi = np.where(fast_range == f)[0][0]
#         mi = np.where(med_range == m)[0][0]
#         si = np.where(slow_range == s)[0][0]
#         cubo_sharpe[fi, mi, si] = sharpe_array_is[idx]

#     # Ordiniamo gli indici dal miglior Sharpe al peggiore
#     valid_indices = np.where(~np.isnan(sharpe_array_is))[0]
#     sorted_valid_indices = valid_indices[np.argsort(sharpe_array_is[valid_indices])[::-1]]
    
#     best_idx = None
    
#     # Iteriamo sui migliori candidati
#     for candidate_idx in sorted_valid_indices:
#         f, m, s = griglia[candidate_idx]
#         candidate_sharpe = sharpe_array_is[candidate_idx]
        
#         # Troviamo le coordinate nel cubo
#         fi = np.where(fast_range == f)[0][0]
#         mi = np.where(med_range == m)[0][0]
#         si = np.where(slow_range == s)[0][0]
        
#         # Estraiamo il vicinato (finestra 3x3x3)
#         fi_min, fi_max = max(0, fi-1), min(shape_3d[0], fi+2)
#         mi_min, mi_max = max(0, mi-1), min(shape_3d[1], mi+2)
#         si_min, si_max = max(0, si-1), min(shape_3d[2], si+2)
        
#         neighborhood = cubo_sharpe[fi_min:fi_max, mi_min:mi_max, si_min:si_max]
        
#         # Rimuoviamo la cella centrale e i NaN per valutare solo i vicini validi
#         neighbors_only = neighborhood.flatten()
#         # Il valore centrale è al centro dell'array appiattito (o lo calcoliamo escludendolo col valore esatto)
#         neighbors_only = neighbors_only[~np.isnan(neighbors_only)]
        
#         # Togliamo il candidato stesso per fare la media dei SOLI vicini
#         neighbors_only = neighbors_only[neighbors_only != candidate_sharpe]
        
#         # Controlliamo se ci sono abbastanza vicini per fare una valutazione
#         if len(neighbors_only) < MIN_VICINI_VALIDI:
#             continue # Scartato: è su un bordo isolato o circondato da NaN
            
#         # Calcoliamo la media dello Sharpe del vicinato
#         avg_neighbor_sharpe = np.mean(neighbors_only)
        
#         # Verifica della Tolleranza: il vicinato deve mantenere almeno il (1 - Tolleranza)% dello Sharpe del candidato
#         soglia_minima_accettabile = candidate_sharpe * (1 - TOLLERANZA_MAX_DEGRADO)
        
#         if avg_neighbor_sharpe >= soglia_minima_accettabile:
#             # TROVATO! È il picco più alto che risiede su un altopiano stabile.
#             best_idx = candidate_idx
#             break
            
#     if best_idx is None:
#         return "NO_STABLE_PLATEAU"
        
#     best_f, best_m, best_s = griglia[best_idx]
#     sharpe_is_scelto = sharpe_array_is[best_idx]
#     data_inizio_oos = close.index[split_idx].strftime('%Y-%m-%d')
    
#     # --- FASE 3: OUT-OF-SAMPLE (40% TEST) ---
#     date_oos = close.iloc[split_idx:].index
#     close_oos = close.iloc[split_idx:].reset_index(drop=True)
    
#     ema1_oos = ema_df[best_f].iloc[split_idx:].reset_index(drop=True)
#     ema2_oos = ema_df[best_m].iloc[split_idx:].reset_index(drop=True)
#     ema3_oos = ema_df[best_s].iloc[split_idx:].reset_index(drop=True)
    
#     entries_oos_raw = (ema1_oos.vbt.crossed_above(ema2_oos) | ema1_oos.vbt.crossed_above(ema3_oos) | ema2_oos.vbt.crossed_above(ema3_oos))
#     exits_oos_raw = (ema1_oos.vbt.crossed_below(ema2_oos) | ema1_oos.vbt.crossed_below(ema3_oos) | ema2_oos.vbt.crossed_below(ema3_oos))
    
#     entries_oos = entries_oos_raw.vbt.signals.fshift(1)
#     exits_oos = exits_oos_raw.vbt.signals.fshift(1)
    
#     pf_oos = vbt.Portfolio.from_signals(
#         close_oos, entries_oos, exits_oos, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     # --- FASE 4: METRICHE STRATEGIA VS BUY & HOLD ---
#     equity_returns_oos = pf_oos.returns()
#     equity_returns_oos.index = date_oos
#     equity_curve_oos = (1 + equity_returns_oos).cumprod()
    
#     giorni_totali_oos = len(equity_curve_oos)
#     anni_totali_oos = giorni_totali_oos / giorni_anno
    
#     cagr_strat = (equity_curve_oos.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if anni_totali_oos > 0 else 0
#     vol_strat = equity_returns_oos.std() * np.sqrt(giorni_anno)
#     sharpe_strat = (equity_returns_oos.mean() / equity_returns_oos.std() * np.sqrt(giorni_anno)) if vol_strat > 0 else 0
    
#     peak_strat = equity_curve_oos.cummax()
#     max_dd_strat = ((equity_curve_oos - peak_strat) / peak_strat).min() if len(equity_curve_oos) > 0 else 0
#     total_trades_oos = pf_oos.trades.count()
    
#     close_oos_bh = close.loc[date_oos]
#     rets_bh = close_oos_bh.pct_change().dropna()
#     equity_bh = (1 + rets_bh).cumprod()
    
#     cagr_bh = (equity_bh.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_bh) > 0 and anni_totali_oos > 0 else 0
#     vol_bh = rets_bh.std() * np.sqrt(giorni_anno)
#     sharpe_bh = (rets_bh.mean() / rets_bh.std() * np.sqrt(giorni_anno)) if vol_bh > 0 else 0
    
#     peak_bh = equity_bh.cummax()
#     max_dd_bh = ((equity_bh - peak_bh) / peak_bh).min() if len(equity_bh) > 0 else 0
    
#     alpha = cagr_strat - cagr_bh
    
#     return {
#         "Param (F-M-S)": f"{best_f}-{best_m}-{best_s}",
#         "Start_OOS": data_inizio_oos,
#         "Shp_IS": sharpe_is_scelto,
#         "Shp_IS_B&H": sharpe_bh_is,
#         "Shp_OOS": sharpe_strat,
#         "Shp_OOS_B&H": sharpe_bh,
#         "CAGR_Str": cagr_strat * 100,
#         "CAGR_B&H": cagr_bh * 100,
#         "Alpha_%": alpha * 100,
#         "DD_Str": max_dd_strat * 100,
#         "DD_B&H": max_dd_bh * 100,
#         "TRD_OOS": total_trades_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"⌛ Split 60-40 per {ticker:<10} | Cluster: {cluster_appartenenza:<12} ... ", end="")
    
#     metrics = ottimizza_asset_split_60_40(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#         print(f"COMPLETATO ✅ (EMA: {metrics['Param (F-M-S)']})")
#     elif metrics == "LOW_SHARPE":
#         print("SCARTATO (Nessun setup valido In-Sample) ⚠️")
#     elif metrics == "NO_STABLE_PLATEAU":
#         print("SCARTATO (Tutti i picchi erano isolati / overfittati) ⚠️")
#     else:
#         print("ERRORE/STORICO INSUFFICIENTE ❌")

# # ==============================================================================
# # 5. LEADERBOARD E STAMPA TABELLONE FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_%", ascending=False).reset_index(drop=True)

#     print("\n" + "="*155)
#     print("🏆 TABELLONE FINALE SPLIT 60/40 3EMA - NEIGHBORHOOD SENSITIVITY (Ordinato per Alpha OOS)")
#     print("="*155)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Start_OOS", "Param (F-M-S)", "Shp_IS", "Shp_IS_B&H", "Shp_OOS", "Shp_OOS_B&H", "CAGR_Str", "CAGR_B&H", "Alpha_%", "DD_Str", "DD_B&H", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "F-M-S", "SHP IS", "SHP IS B&H", "SHP OOS", "SHP OOS B&H", "CAGR STR", "CAGR B&H", "ALPHA %", "DD STR", "DD B&H", "TRD"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':       '{:,.2f}'.format,
#             'SHP IS B&H':   '{:,.2f}'.format,
#             'SHP OOS':      '{:,.2f}'.format,
#             'SHP OOS B&H':  '{:,.2f}'.format,
#             'CAGR STR':     '{:,.1f}%'.format,
#             'CAGR B&H':     '{:,.1f}%'.format,
#             'ALPHA %':      '{:,.1f}%'.format,
#             'DD STR':       '{:,.1f}%'.format,
#             'DD B&H':       '{:,.1f}%'.format,
#             'TRD':          '{:,}'.format
#         }
#     ))
#     print("="*155)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 (Neighborhood Sensitivity Analysis) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.
⌛ Split 60-40 per US100.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 8-56-150)
⌛ Split 60-40 per US30.cash  | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 20-68-160)
⌛ Split 60-40 per US500.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 10-28-240)
⌛ Split 60-40 per BNBUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-180)
⌛ Split 60-40 per BTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 26-32-120)
⌛ Split 60-40 per ETHUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-20-140)
⌛ Split 60-40 per GRTUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 18-36-155)
⌛ Split 60-40 per LTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 8-24-170)
⌛ Split 60-40 per MANUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-28-190)
⌛ Split 60-40 per NERUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 16-24-120)
⌛ 

# three split dataset: is + val + oos

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Three-Way Split (50% IS, 25% VAL, 25% OOS) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: THREE-WAY SPLIT ENGINE
# # ==============================================================================

# def ottimizza_asset_tre_vie(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
    
#     # ---------------------------------------------------------
#     # CALCOLO DEGLI INDICI DI TAGLIO (50% / 25% / 25%)
#     # ---------------------------------------------------------
#     idx_val = int(total_bars * 0.50)  # Fine IS, Inizio VAL
#     idx_oos = int(total_bars * 0.75)  # Fine VAL, Inizio OOS
    
#     # Check di sicurezza: abbiamo abbastanza dati in ogni blocco?
#     if idx_val < 150 or (idx_oos - idx_val) < 50 or (total_bars - idx_oos) < 50:
#         return None
        
#     f_arr = np.array([t[0] for t in griglia])
#     m_arr = np.array([t[1] for t in griglia])
#     s_arr = np.array([t[2] for t in griglia])
#     multi_idx = pd.MultiIndex.from_arrays([f_arr, m_arr, s_arr], names=['Fast', 'Med', 'Slow'])
    
#     # ==========================================================================
#     # --- FASE 1: IN-SAMPLE (50% TRAIN)
#     # ==========================================================================
#     close_is = close.iloc[:idx_val].reset_index(drop=True)
#     ema_is = ema_df.iloc[:idx_val].reset_index(drop=True)
    
#     ema1_mat = ema_is[f_arr]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_is[m_arr]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_is[s_arr]; ema3_mat.columns = multi_idx
    
#     entries_is = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat)).vbt.signals.fshift(1)
#     exits_is = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat)).vbt.signals.fshift(1)
    
#     pf_is = vbt.Portfolio.from_signals(close_is, entries_is, exits_is, freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005)
    
#     rets_is = pf_is.returns()
#     std_is = rets_is.std()
    
#     anni_is = len(close_is) / giorni_anno
#     tpy_is_array = pf_is.trades.count().reindex(multi_idx, fill_value=0).values / anni_is if anni_is > 0 else 0
    
#     sharpe_is = np.where((std_is > 0), (rets_is.mean() / std_is) * np.sqrt(giorni_anno), 0.0)
    
#     # ==========================================================================
#     # --- FASE 2: VALIDATION SET (25% VALIDAZIONE CROSS)
#     # ==========================================================================
#     close_val = close.iloc[idx_val:idx_oos].reset_index(drop=True)
#     ema_val = ema_df.iloc[idx_val:idx_oos].reset_index(drop=True)
    
#     ema1_val = ema_val[f_arr]; ema1_val.columns = multi_idx
#     ema2_val = ema_val[m_arr]; ema2_val.columns = multi_idx
#     ema3_val = ema_val[s_arr]; ema3_val.columns = multi_idx
    
#     entries_val = (ema1_val.vbt.crossed_above(ema2_val) | ema1_val.vbt.crossed_above(ema3_val) | ema2_val.vbt.crossed_above(ema3_val)).vbt.signals.fshift(1)
#     exits_val = (ema1_val.vbt.crossed_below(ema2_val) | ema1_val.vbt.crossed_below(ema3_val) | ema2_val.vbt.crossed_below(ema3_val)).vbt.signals.fshift(1)
    
#     pf_val = vbt.Portfolio.from_signals(close_val, entries_val, exits_val, freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005)
    
#     rets_val = pf_val.returns()
#     std_val = rets_val.std()
#     sharpe_val = np.where((std_val > 0), (rets_val.mean() / std_val) * np.sqrt(giorni_anno), 0.0)
    
#     # ---------------------------------------------------------
#     # LA SELEZIONE (ROBUSTNESS SCORE IS + VAL)
#     # ---------------------------------------------------------
#     # Un parametro è selezionabile solo se fa almeno 2 trade all'anno in IS e ha uno Sharpe IS decente (>0.5)
#     mask_valida = (tpy_is_array >= 2) & (sharpe_is > 0.5)
    
#     if not np.any(mask_valida):
#         return "NO_VALID_IS"
        
#     # Calcoliamo lo Score solo per i parametri validi. 
#     # Score = Media degli Sharpe (IS e VAL) - Penalità per la differenza (Vogliamo coerenza)
#     score_robusto = ((sharpe_is + sharpe_val) / 2) - np.abs(sharpe_is - sharpe_val)
#     score_robusto[~mask_valida] = -99.0
    
#     best_idx = np.argmax(score_robusto)
#     best_f, best_m, best_s = griglia[best_idx]
    
#     shp_is_scelto = sharpe_is[best_idx]
#     shp_val_scelto = sharpe_val[best_idx]
    
#     data_inizio_val = close.index[idx_val].strftime('%Y-%m-%d')
#     data_inizio_oos = close.index[idx_oos].strftime('%Y-%m-%d')
    
#     # ==========================================================================
#     # --- FASE 3: OUT-OF-SAMPLE (25% TEST - L'ESAME FINALE)
#     # ==========================================================================
#     date_oos = close.iloc[idx_oos:].index
#     close_oos = close.iloc[idx_oos:].reset_index(drop=True)
    
#     ema1_oos = ema_df[best_f].iloc[idx_oos:].reset_index(drop=True)
#     ema2_oos = ema_df[best_m].iloc[idx_oos:].reset_index(drop=True)
#     ema3_oos = ema_df[best_s].iloc[idx_oos:].reset_index(drop=True)
    
#     entries_oos_raw = (ema1_oos.vbt.crossed_above(ema2_oos) | ema1_oos.vbt.crossed_above(ema3_oos) | ema2_oos.vbt.crossed_above(ema3_oos))
#     exits_oos_raw = (ema1_oos.vbt.crossed_below(ema2_oos) | ema1_oos.vbt.crossed_below(ema3_oos) | ema2_oos.vbt.crossed_below(ema3_oos))
    
#     entries_oos = entries_oos_raw.vbt.signals.fshift(1)
#     exits_oos = exits_oos_raw.vbt.signals.fshift(1)
    
#     pf_oos = vbt.Portfolio.from_signals(close_oos, entries_oos, exits_oos, freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005)
    
#     # Calcolo Metriche Strategia OOS
#     equity_returns_oos = pf_oos.returns()
#     equity_returns_oos.index = date_oos
#     equity_curve_oos = (1 + equity_returns_oos).cumprod()
    
#     giorni_totali_oos = len(equity_curve_oos)
#     anni_totali_oos = giorni_totali_oos / giorni_anno
    
#     cagr_strat = (equity_curve_oos.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_curve_oos) > 0 and anni_totali_oos > 0 else 0
#     vol_strat = equity_returns_oos.std() * np.sqrt(giorni_anno)
#     sharpe_oos = (equity_returns_oos.mean() / equity_returns_oos.std() * np.sqrt(giorni_anno)) if vol_strat > 0 else 0
    
#     peak_strat = equity_curve_oos.cummax()
#     max_dd_strat = ((equity_curve_oos - peak_strat) / peak_strat).min() if len(equity_curve_oos) > 0 else 0
#     total_trades_oos = pf_oos.trades.count()
    
#     # Calcolo Buy & Hold sullo stesso identico periodo OOS
#     close_oos_bh = close.loc[date_oos]
#     rets_bh = close_oos_bh.pct_change().dropna()
#     equity_bh = (1 + rets_bh).cumprod()
    
#     cagr_bh = (equity_bh.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_bh) > 0 and anni_totali_oos > 0 else 0
#     vol_bh = rets_bh.std() * np.sqrt(giorni_anno)
#     sharpe_bh = (rets_bh.mean() / rets_bh.std() * np.sqrt(giorni_anno)) if vol_bh > 0 else 0
    
#     peak_bh = equity_bh.cummax()
#     max_dd_bh = ((equity_bh - peak_bh) / peak_bh).min() if len(equity_bh) > 0 else 0
    
#     alpha = cagr_strat - cagr_bh
    
#     return {
#         "Param (F-M-S)": f"{best_f}-{best_m}-{best_s}",
#         "Inizio_VAL": data_inizio_val,
#         "Inizio_OOS": data_inizio_oos,
#         "Shp_IS": shp_is_scelto,
#         "Shp_VAL": shp_val_scelto,
#         "Shp_OOS": sharpe_oos,
#         "Shp_B&H": sharpe_bh,
#         "CAGR_Str": cagr_strat * 100,
#         "CAGR_B&H": cagr_bh * 100,
#         "Alpha_%": alpha * 100,
#         "DD_Str": max_dd_strat * 100,
#         "DD_B&H": max_dd_bh * 100,
#         "TRD_OOS": total_trades_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"⌛ 3-Way Split per {ticker:<10} | Cluster: {cluster_appartenenza:<12} ... ", end="")
    
#     metrics = ottimizza_asset_tre_vie(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#         print(f"COMPLETATO ✅ (EMA: {metrics['Param (F-M-S)']})")
#     elif metrics == "NO_VALID_IS":
#         print("SCARTATO (Nessun setup valido nell'In-Sample) ⚠️")
#     else:
#         print("ERRORE/STORICO INSUFFICIENTE ❌")

# # ==============================================================================
# # 5. LEADERBOARD E STAMPA TABELLONE FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     # Ordiniamo per Alpha OOS: la verità assoluta del mercato reale
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_%", ascending=False).reset_index(drop=True)

#     print("\n" + "="*160)
#     print("🏆 TABELLONE FINALE 3-WAY SPLIT (Ottimizzazione su IS, Selezione su VAL, Test su OOS)")
#     print("="*160)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Inizio_OOS", "Param (F-M-S)", "Shp_IS", "Shp_VAL", "Shp_OOS", "Shp_B&H", "CAGR_Str", "CAGR_B&H", "Alpha_%", "DD_Str", "DD_B&H", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "F-M-S", "SHP IS", "SHP VAL", "SHP OOS", "SHP B&H", "CAGR STR", "CAGR B&H", "ALPHA %", "DD STR", "DD B&H", "TRD"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'SHP VAL':   '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'SHP B&H':   '{:,.2f}'.format,
#             'CAGR STR':  '{:,.1f}%'.format,
#             'CAGR B&H':  '{:,.1f}%'.format,
#             'ALPHA %':   '{:,.1f}%'.format,
#             'DD STR':    '{:,.1f}%'.format,
#             'DD B&H':    '{:,.1f}%'.format,
#             'TRD':       '{:,}'.format
#         }
#     ))
#     print("="*160)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Three-Way Split (50% IS, 25% VAL, 25% OOS) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.
⌛ 3-Way Split per US100.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 4-60-230)
⌛ 3-Way Split per US30.cash  | Cluster: cash_cfd     ... SCARTATO (Nessun setup valido nell'In-Sample) ⚠️
⌛ 3-Way Split per US500.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (EMA: 16-28-155)
⌛ 3-Way Split per BNBUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-44-160)
⌛ 3-Way Split per BTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 6-44-130)
⌛ 3-Way Split per ETHUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 4-68-140)
⌛ 3-Way Split per GRTUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 18-36-150)
⌛ 3-Way Split per LTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 8-44-210)
⌛ 3-Way Split per MANUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 6-48-220)
⌛ 3-Way Split per NERUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (EMA: 

# 60-40 + ensemble

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# import gc  # Memory Management

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 (Ensemble Top 3 + Sensitivity 25%) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: SPLIT 60/40 + SENSITIVITY + ENSEMBLE ENGINE
# # ==============================================================================

# TOLLERANZA_MAX_DEGRADO = 0.25 
# MIN_VICINI_VALIDI = 6         

# def calc_advanced_metrics(rets, giorni_anno):
#     """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
#     if len(rets) == 0 or rets.isnull().all():
#         return 0, 0, 0, 0, 0
        
#     # CAGR
#     anni = len(rets) / giorni_anno
#     cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
#     # Volatilità e Sharpe (Annualizzati)
#     daily_std = rets.std()
#     vol = daily_std * np.sqrt(giorni_anno)
#     sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
#     # Sortino (Annualizzato)
#     neg_rets = rets[rets < 0]
#     daily_down_std = neg_rets.std()
#     sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
#     # Max DD
#     cum = (1 + rets).cumprod()
#     peak = cum.cummax()
#     dd = (cum - peak) / peak
#     max_dd = dd.min() if len(dd) > 0 else 0
    
#     return cagr, vol, sharpe, sortino, max_dd

# def ottimizza_split_60_40_ensemble(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_cols = [t[0] for t in griglia]
#     m_cols = [t[1] for t in griglia]
#     s_cols = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
#     ema1_mat = ema_df[f_cols]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_df[m_cols]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_df[s_cols]; ema3_mat.columns = multi_idx
    
#     entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
#     exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
#     entries_global = entries_raw.vbt.signals.fshift(1)
#     exits_global = exits_raw.vbt.signals.fshift(1)
    
#     # Liberiamo la RAM subito
#     del ema1_mat, ema2_mat, ema3_mat, entries_raw, exits_raw
#     gc.collect()
    
#     # DATI IS e OOS
#     close_is = close.iloc[:split_idx]
#     close_oos = close.iloc[split_idx:]
#     data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    
#     # --- CALCOLO BUY & HOLD IS/OOS ---
#     rets_bh_is = close_is.pct_change().dropna()
#     cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
#     rets_bh_oos = close_oos.pct_change().dropna()
#     cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)
    
#     # --- IN-SAMPLE STRATEGY CALCULATION ---
#     pf_is_all = vbt.Portfolio.from_signals(close_is, entries_global.iloc[:split_idx], exits_global.iloc[:split_idx], freq=None)
#     rets_is_all = pf_is_all.returns()
#     std_is_all = rets_is_all.std()
    
#     anni_is = len(close_is) / giorni_anno
#     tpy_is_array = pf_is_all.trades.count().values / anni_is if anni_is > 0 else 0
    
#     # RICHIEDIAMO ALMENO 2 TRADE ANNUI NELL'IN-SAMPLE
#     sharpe_array_is = np.where(
#         (std_is_all > 0) & (tpy_is_array >= 2), 
#         (rets_is_all.mean() / std_is_all) * np.sqrt(giorni_anno), 
#         np.nan
#     )
    
#     top_stable_params = []
    
#     if not np.isnan(sharpe_array_is).all():
#         shape_3d = (len(fast_range), len(med_range), len(slow_range))
#         cubo_sharpe = np.full(shape_3d, np.nan)
#         for idx, (f, m, s) in enumerate(griglia):
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             cubo_sharpe[fi, mi, si] = sharpe_array_is[idx]

#         valid_indices = np.where(~np.isnan(sharpe_array_is))[0]
#         sorted_valid_indices = valid_indices[np.argsort(sharpe_array_is[valid_indices])[::-1]]
        
#         for candidate_idx in sorted_valid_indices:
#             f, m, s = griglia[candidate_idx]
#             candidate_sharpe = sharpe_array_is[candidate_idx]
            
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
            
#             fi_min, fi_max = max(0, fi-1), min(shape_3d[0], fi+2)
#             mi_min, mi_max = max(0, mi-1), min(shape_3d[1], mi+2)
#             si_min, si_max = max(0, si-1), min(shape_3d[2], si+2)
            
#             neighborhood = cubo_sharpe[fi_min:fi_max, mi_min:mi_max, si_min:si_max].flatten()
#             neighbors_only = neighborhood[~np.isnan(neighborhood)]
#             neighbors_only = neighbors_only[neighbors_only != candidate_sharpe]
            
#             if len(neighbors_only) < MIN_VICINI_VALIDI:
#                 continue
                
#             avg_neighbor_sharpe = np.mean(neighbors_only)
#             soglia_minima = candidate_sharpe * (1 - TOLLERANZA_MAX_DEGRADO)
            
#             if avg_neighbor_sharpe >= soglia_minima:
#                 top_stable_params.append((f, m, s))
#                 if len(top_stable_params) == 3: # Cerchiamo le Top 3
#                     break

#     del pf_is_all, rets_is_all, std_is_all
#     if 'cubo_sharpe' in locals(): del cubo_sharpe
#     gc.collect()

#     # --- ENSEMBLE CALCULATION ---
#     if not top_stable_params:
#         return "LOW_SHARPE"
        
#     is_rets_list, oos_rets_list = [], []
#     is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
#     for p_tuple in top_stable_params:
#         # IS singolo parametro
#         pf_i = vbt.Portfolio.from_signals(close_is, entries_global[p_tuple].iloc[:split_idx], exits_global[p_tuple].iloc[:split_idx], freq=None)
#         is_rets_list.append(pf_i.returns())
#         is_trd.append(pf_i.trades.count())
#         is_win.append((pf_i.trades.pnl > 0).sum())
        
#         # OOS singolo parametro
#         pf_o = vbt.Portfolio.from_signals(close_oos, entries_global[p_tuple].iloc[split_idx:], exits_global[p_tuple].iloc[split_idx:], freq=None)
#         oos_rets_list.append(pf_o.returns())
#         oos_trd.append(pf_o.trades.count())
#         oos_win.append((pf_o.trades.pnl > 0).sum())
        
#     # Blending (Media equiponderata al 33%)
#     blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
#     blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
#     # Metriche Ensemble IS
#     cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
#     avg_trd_is = np.mean(is_trd)
#     win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
#     # Metriche Ensemble OOS
#     cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
#     avg_trd_oos = np.mean(oos_trd)
#     win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
#     alpha_oos = cagr_oos - cagr_bh_oos
#     param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
#     # Pulizia memoria finale asset
#     del entries_global, exits_global, ema_df
#     gc.collect()

#     return {
#         "Parametri": param_str,
#         "Inizio_OOS": data_inizio_oos,
#         "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
#         "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
#         "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
#         "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
#         "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
#         "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
#         "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
#         "Alpha_OOS": alpha_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"\n============================================================")
#     print(f"⌛ ELABORAZIONE {ticker:<8} | Cluster: {cluster_appartenenza}")
    
#     metrics = ottimizza_split_60_40_ensemble(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         # Stampa a video dettagliata per singolo ticker
#         print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
#         print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
#         print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
#         print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
#         print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
#         print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#     elif metrics == "LOW_SHARPE":
#         print("⚠️ SCARTATO: Nessun altopiano stabile trovato In-Sample.")
#     else:
#         print("❌ ERRORE: Storico dati insufficiente.")

# # ==============================================================================
# # 5. LEADERBOARD FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

#     print("\n" + "="*160)
#     print("🏆 TABELLONE FINALE SPLIT 60/40 ENSEMBLE (Ordinato per Alpha OOS)")
#     print("="*160)

#     # Selezioniamo le colonne più importanti per il tabellone globale per non saturare lo schermo
#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
#         "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
#         "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Ensemble)", 
#         "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
#         "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'B&H IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'B&H OOS':   '{:,.2f}'.format,
#             'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
#             'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
#             'ALPHA %':   lambda x: f"{x*100:,.1f}%",
#             'DD OOS':    lambda x: f"{x*100:,.1f}%",
#             'B&H DD':    lambda x: f"{x*100:,.1f}%",
#             'TRD OOS':   '{:,.1f}'.format
#         }
#     ))
#     print("="*160)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 (Ensemble Top 3 + Sensitivity 25%) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.

⌛ ELABORAZIONE US100.cash | Cluster: cash_cfd
✅ Parametri Scelti: [(8,56,150), (4,60,230), (4,60,150)] | Inizio OOS: 2024-04-09
  [IS]  STRAT -> Shp: 1.10 | Cagr: 14.1% | DD: -11.0% | Sort: 1.38 | Trd: 10.7 | Win: 56.2%
  [IS]  B&H   -> Shp: 0.51 | Cagr: 9.5% | DD: -35.6% | Sort: 0.74
  [OOS] STRAT -> Shp: 1.42 | Cagr: 20.7% | DD: -12.2% | Sort: 1.57 | Trd: 7.3 | Win: 54.5%
  [OOS] B&H   -> Shp: 1.21 | Cagr: 25.9% | DD: -22.8% | Sort: 1.65
  🔥 ALPHA OOS:  -5.21%

⌛ ELABORAZIONE US30.cash | Cluster: cash_cfd
✅ Parametri Scelti: [(22,64,115), (16,80,100), (26,56,180)] | Inizio OOS: 2023-06-28
  [IS]  STRAT -> Shp: 0.43 | Cagr: 4.5% | DD: -14.7% | Sort: 0.42 | Trd: 10.0 | Win: 43.3%
  [IS]  B&H   -> Shp: 0.42 | Cagr: 6.9% | DD: -37.1% | Sort: 0.50
  [OOS] STRAT -> Shp: 0.99 | Cagr: 9.5% | DD: -10.6% | Sort: 1.28 | Trd: 4.7 | Win: 78.6%
  [OOS] B&H   -> Shp: 

# conv 3d + ensemble

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# import gc  # Memory Management

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 (Convoluzione 3D + Ensemble Top 3) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# # ==============================================================================
# fast_range = np.arange(4, 42, 2)     
# med_range  = np.arange(20, 100, 4)   
# slow_range = np.arange(100, 250, 5)  

# griglia_parametri = [
#     (f, m, s) 
#     for f in fast_range for m in med_range for s in slow_range 
#     if f < m < s
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: SPLIT 60/40 + CONVOLUZIONE 3D + ENSEMBLE
# # ==============================================================================

# MIN_VICINI_VALIDI = 6         

# def calc_advanced_metrics(rets, giorni_anno):
#     """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
#     if len(rets) == 0 or rets.isnull().all():
#         return 0, 0, 0, 0, 0
        
#     # CAGR
#     anni = len(rets) / giorni_anno
#     cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
#     # Volatilità e Sharpe (Annualizzati)
#     daily_std = rets.std()
#     vol = daily_std * np.sqrt(giorni_anno)
#     sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
#     # Sortino (Annualizzato)
#     neg_rets = rets[rets < 0]
#     daily_down_std = neg_rets.std()
#     sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
#     # Max DD
#     cum = (1 + rets).cumprod()
#     peak = cum.cummax()
#     dd = (cum - peak) / peak
#     max_dd = dd.min() if len(dd) > 0 else 0
    
#     return cagr, vol, sharpe, sortino, max_dd

# def ottimizza_split_60_40_ensemble(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
#     ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_cols = [t[0] for t in griglia]
#     m_cols = [t[1] for t in griglia]
#     s_cols = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
#     ema1_mat = ema_df[f_cols]; ema1_mat.columns = multi_idx
#     ema2_mat = ema_df[m_cols]; ema2_mat.columns = multi_idx
#     ema3_mat = ema_df[s_cols]; ema3_mat.columns = multi_idx
    
#     entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
#     exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
#     entries_global = entries_raw.vbt.signals.fshift(1)
#     exits_global = exits_raw.vbt.signals.fshift(1)
    
#     del ema1_mat, ema2_mat, ema3_mat, entries_raw, exits_raw
#     gc.collect()
    
#     # DATI IS e OOS
#     close_is = close.iloc[:split_idx]
#     close_oos = close.iloc[split_idx:]
#     data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    
#     # --- CALCOLO BUY & HOLD IS/OOS ---
#     rets_bh_is = close_is.pct_change().dropna()
#     cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
#     rets_bh_oos = close_oos.pct_change().dropna()
#     cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)
    
#     # --- IN-SAMPLE STRATEGY CALCULATION ---
#     pf_is_all = vbt.Portfolio.from_signals(close_is, entries_global.iloc[:split_idx], exits_global.iloc[:split_idx], freq=None)
#     rets_is_all = pf_is_all.returns()
#     std_is_all = rets_is_all.std()
    
#     anni_is = len(close_is) / giorni_anno
#     tpy_is_array = pf_is_all.trades.count().values / anni_is if anni_is > 0 else 0
    
#     sharpe_array_is = np.where(
#         (std_is_all > 0) & (tpy_is_array >= 2), 
#         (rets_is_all.mean() / std_is_all) * np.sqrt(giorni_anno), 
#         np.nan
#     )
    
#     top_stable_params = []
    
#     if not np.isnan(sharpe_array_is).all():
#         shape_3d = (len(fast_range), len(med_range), len(slow_range))
#         cubo_sharpe = np.full(shape_3d, np.nan)
#         for idx, (f, m, s) in enumerate(griglia):
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             cubo_sharpe[fi, mi, si] = sharpe_array_is[idx]

#         valid_indices = np.where(~np.isnan(sharpe_array_is))[0]
        
#         convolved_candidates = []
        
#         # 🔥 FASE DI CONVOLUZIONE 3D 🔥
#         for candidate_idx in valid_indices:
#             f, m, s = griglia[candidate_idx]
            
#             fi = np.where(fast_range == f)[0][0]
#             mi = np.where(med_range == m)[0][0]
#             si = np.where(slow_range == s)[0][0]
            
#             # Estrazione del cubo 3x3x3 (Vicinato + Cella Centrale)
#             fi_min, fi_max = max(0, fi-1), min(shape_3d[0], fi+2)
#             mi_min, mi_max = max(0, mi-1), min(shape_3d[1], mi+2)
#             si_min, si_max = max(0, si-1), min(shape_3d[2], si+2)
            
#             neighborhood = cubo_sharpe[fi_min:fi_max, mi_min:mi_max, si_min:si_max].flatten()
#             valid_neighbors = neighborhood[~np.isnan(neighborhood)]
            
#             if len(valid_neighbors) >= MIN_VICINI_VALIDI:
#                 # Score Convoluzione = Media(Sharpe) - DeviazioneStandard(Sharpe)
#                 # Premia gli altopiani altissimi e lisci, penalizza i picchi nervosi e isolati
#                 conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
#                 convolved_candidates.append((conv_score, (f, m, s)))
                
#         # Ordiniamo tutti i candidati in base al loro Score di Convoluzione
#         convolved_candidates.sort(key=lambda x: x[0], reverse=True)
        
#         # Selezioniamo matematicamente i 3 migliori in assoluto
#         top_stable_params = [x[1] for x in convolved_candidates[:3]]

#     del pf_is_all, rets_is_all, std_is_all
#     if 'cubo_sharpe' in locals(): del cubo_sharpe
#     gc.collect()

#     # --- ENSEMBLE CALCULATION ---
#     if not top_stable_params:
#         return "LOW_SHARPE"
        
#     is_rets_list, oos_rets_list = [], []
#     is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
#     for p_tuple in top_stable_params:
#         # IS singolo parametro
#         pf_i = vbt.Portfolio.from_signals(close_is, entries_global[p_tuple].iloc[:split_idx], exits_global[p_tuple].iloc[:split_idx], freq=None)
#         is_rets_list.append(pf_i.returns())
#         is_trd.append(pf_i.trades.count())
#         is_win.append((pf_i.trades.pnl > 0).sum())
        
#         # OOS singolo parametro
#         pf_o = vbt.Portfolio.from_signals(close_oos, entries_global[p_tuple].iloc[split_idx:], exits_global[p_tuple].iloc[split_idx:], freq=None)
#         oos_rets_list.append(pf_o.returns())
#         oos_trd.append(pf_o.trades.count())
#         oos_win.append((pf_o.trades.pnl > 0).sum())
        
#     # Blending (Media equiponderata al 33%)
#     blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
#     blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
#     # Metriche Ensemble IS
#     cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
#     avg_trd_is = np.mean(is_trd)
#     win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
#     # Metriche Ensemble OOS
#     cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
#     avg_trd_oos = np.mean(oos_trd)
#     win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
#     alpha_oos = cagr_oos - cagr_bh_oos
#     param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
#     del entries_global, exits_global, ema_df
#     gc.collect()

#     return {
#         "Parametri": param_str,
#         "Inizio_OOS": data_inizio_oos,
#         "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
#         "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
#         "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
#         "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
#         "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
#         "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
#         "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
#         "Alpha_OOS": alpha_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"\n============================================================")
#     print(f"⌛ ELABORAZIONE {ticker:<8} | Cluster: {cluster_appartenenza}")
    
#     metrics = ottimizza_split_60_40_ensemble(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
#         print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
#         print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
#         print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
#         print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
#         print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#     elif metrics == "LOW_SHARPE":
#         print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
#     else:
#         print("❌ ERRORE: Storico dati insufficiente.")

# # ==============================================================================
# # 5. LEADERBOARD FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

#     print("\n" + "="*160)
#     print("🏆 TABELLONE FINALE SPLIT 60/40 (CONVOLUZIONE 3D + ENSEMBLE)")
#     print("="*160)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
#         "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
#         "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Ensemble)", 
#         "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
#         "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'B&H IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'B&H OOS':   '{:,.2f}'.format,
#             'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
#             'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
#             'ALPHA %':   lambda x: f"{x*100:,.1f}%",
#             'DD OOS':    lambda x: f"{x*100:,.1f}%",
#             'B&H DD':    lambda x: f"{x*100:,.1f}%",
#             'TRD OOS':   '{:,.1f}'.format
#         }
#     ))
#     print("="*160)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 (Convoluzione 3D + Ensemble Top 3) su 48 asset.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.

⌛ ELABORAZIONE US100.cash | Cluster: cash_cfd
✅ Parametri Scelti: [(4,56,155), (4,60,155), (4,56,150)] | Inizio OOS: 2024-04-09
  [IS]  STRAT -> Shp: 0.99 | Cagr: 12.9% | DD: -13.7% | Sort: 1.25 | Trd: 12.3 | Win: 51.4%
  [IS]  B&H   -> Shp: 0.51 | Cagr: 9.5% | DD: -35.6% | Sort: 0.74
  [OOS] STRAT -> Shp: 1.48 | Cagr: 21.8% | DD: -11.5% | Sort: 1.63 | Trd: 8.7 | Win: 61.5%
  [OOS] B&H   -> Shp: 1.21 | Cagr: 25.9% | DD: -22.8% | Sort: 1.65
  🔥 ALPHA OOS:  -4.09%

⌛ ELABORAZIONE US30.cash | Cluster: cash_cfd
✅ Parametri Scelti: [(22,64,160), (18,76,115), (18,76,105)] | Inizio OOS: 2023-06-28
  [IS]  STRAT -> Shp: 0.44 | Cagr: 4.6% | DD: -14.6% | Sort: 0.43 | Trd: 10.0 | Win: 43.3%
  [IS]  B&H   -> Shp: 0.42 | Cagr: 6.9% | DD: -37.1% | Sort: 0.50
  [OOS] STRAT -> Shp: 1.00 | Cagr: 9.7% | DD: -10.4% | Sort: 1.31 | Trd: 5.0 | Win: 80.0%
  [OOS] B&H   -> Shp: 

# ensemble + conv 3d con distanza minima

In [2]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CON AUTO-RECOVER DA PARQUET)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Ricerca flessibile per evitare KeyError se i nomi dei Tier nel JSON sono estesi
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# --- PROTEZIONE DA NAMEERROR: SE LA RAM È VUOTA, LEGGE DIRETTAMENTE I PARQUET TARGET ---
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    # Se i dati sono già pronti in RAM, usiamo quelli per massima velocità
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        # Carichiamo solo i file dei ticker inclusi nella mappa strategie
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore Split 60/40 Asincrono (Convoluzione 3D + Ensemble) su {len(all_tickers_dict)} asset indipendenti.")

# ==============================================================================
# 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE
# ==============================================================================
fast_range = np.arange(4, 42, 2)     
med_range  = np.arange(20, 100, 4)   
slow_range = np.arange(100, 250, 5)  

griglia_parametri = [
    (f, m, s) 
    for f in fast_range for m in med_range for s in slow_range 
    for f_val, m_val, s_val in [(int(f), int(m), int(s))]
    if f_val < m_val < s_val
]
print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 ASINCRONO + CONVOLUZIONE + ENSEMBLE ENGINE
# ==============================================================================

MIN_VICINI_VALIDI = 6         

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
        
    # CAGR
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
    # Volatilità e Sharpe (Annualizzati)
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
    # Sortino (Annualizzato)
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
    # Max DD
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    
    return cagr, vol, sharpe, sortino, max_dd

def ottimizza_split_60_40_ensemble(ticker, df, griglia, is_crypto=False):
    close = df['Close'].copy()
    close.index = pd.to_datetime(close.index)
    
    finestre_uniche = sorted(list(set([p for triplet in griglia for p in triplet])))
    ema_df = pd.DataFrame({w: vbt.MA.run(close, window=w, ewm=True).ma for w in finestre_uniche})
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    f_cols = [t[0] for t in griglia]
    m_cols = [t[1] for t in griglia]
    s_cols = [t[2] for t in griglia]
    multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Med', 'Slow'])
    
    ema1_mat = ema_df[f_cols].copy(); ema1_mat.columns = multi_idx
    ema2_mat = ema_df[m_cols].copy(); ema2_mat.columns = multi_idx
    ema3_mat = ema_df[s_cols].copy(); ema3_mat.columns = multi_idx
    
    entries_raw = (ema1_mat.vbt.crossed_above(ema2_mat) | ema1_mat.vbt.crossed_above(ema3_mat) | ema2_mat.vbt.crossed_above(ema3_mat))
    exits_raw = (ema1_mat.vbt.crossed_below(ema2_mat) | ema1_mat.vbt.crossed_below(ema3_mat) | ema2_mat.vbt.crossed_below(ema3_mat))
    
    entries_global = entries_raw.vbt.signals.fshift(1)
    exits_global = exits_raw.vbt.signals.fshift(1)
    
    del ema1_mat, ema2_mat, ema3_mat, entries_raw, exits_raw
    gc.collect()
    
    # DATI IS e OOS INDIPENDENTI PER QUESTO PARQUET ASINCRONO
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)
    
    # --- IN-SAMPLE STRATEGY CALCULATION (Forzato freq='D' per asincronia) ---
    pf_is_all = vbt.Portfolio.from_signals(close_is, entries_global.iloc[:split_idx], exits_global.iloc[:split_idx], freq='D')
    rets_is_all = pf_is_all.returns()
    std_is_all = rets_is_all.std()
    
    anni_is = len(close_is) / giorni_anno
    tpy_is_array = pf_is_all.trades.count().values / anni_is if anni_is > 0 else 0
    
    sharpe_array_is = np.where(
        (std_is_all > 0) & (tpy_is_array >= 2), 
        (rets_is_all.mean() / std_is_all) * np.sqrt(giorni_anno), 
        np.nan
    )
    
    top_stable_params = []
    
    if not np.isnan(sharpe_array_is).all():
        shape_3d = (len(fast_range), len(med_range), len(slow_range))
        cubo_sharpe = np.full(shape_3d, np.nan)
        for idx, (f, m, s) in enumerate(griglia):
            fi = np.where(fast_range == f)[0][0]
            mi = np.where(med_range == m)[0][0]
            si = np.where(slow_range == s)[0][0]
            cubo_sharpe[fi, mi, si] = sharpe_array_is[idx]

        valid_indices = np.where(~np.isnan(sharpe_array_is))[0]
        convolved_candidates = []
        
        # FASE DI CONVOLUZIONE 3D
        for candidate_idx in valid_indices:
            f, m, s = griglia[candidate_idx]
            fi = np.where(fast_range == f)[0][0]
            mi = np.where(med_range == m)[0][0]
            si = np.where(slow_range == s)[0][0]
            
            fi_min, fi_max = max(0, fi-1), min(shape_3d[0], fi+2)
            mi_min, mi_max = max(0, mi-1), min(shape_3d[1], mi+2)
            si_min, si_max = max(0, si-1), min(shape_3d[2], si+2)
            
            neighborhood = cubo_sharpe[fi_min:fi_max, mi_min:mi_max, si_min:si_max].flatten()
            valid_neighbors = neighborhood[~np.isnan(neighborhood)]
            
            if len(valid_neighbors) >= MIN_VICINI_VALIDI:
                conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                convolved_candidates.append((conv_score, (f, m, s)))
                
        convolved_candidates.sort(key=lambda x: x[0], reverse=True)
        
        # DISTANZIAMENTO SPAZIALE + LOGICA PALI DELLA PORTA
        DISTANZA_MINIMA = 20 
        
        for score, params in convolved_candidates:
            f, m, s = params
            troppo_vicino = False
            
            for (scelto_f, scelto_m, scelto_s) in top_stable_params:
                distanza_totale = abs(f - scelto_f) + abs(m - scelto_m) + abs(s - scelto_s)
                if distanza_totale < DISTANZA_MINIMA:
                    troppo_vicino = True
                    break 
                if f == scelto_f and s == scelto_s:
                    troppo_vicino = True
                    break
            
            if not troppo_vicino:
                top_stable_params.append((f, m, s))
                if len(top_stable_params) == 3: 
                    break

    del pf_is_all, rets_is_all, std_is_all
    if 'cubo_sharpe' in locals(): del cubo_sharpe
    gc.collect()

    # --- ENSEMBLE CALCULATION (Forzato freq='D' per asincronia) ---
    if not top_stable_params:
        return "LOW_SHARPE"
        
    is_rets_list, oos_rets_list = [], []
    is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
    for p_tuple in top_stable_params:
        pf_i = vbt.Portfolio.from_signals(close_is, entries_global[p_tuple].iloc[:split_idx], exits_global[p_tuple].iloc[:split_idx], freq='D')
        is_rets_list.append(pf_i.returns())
        is_trd.append(pf_i.trades.count())
        is_win.append((pf_i.trades.pnl > 0).sum())
        
        pf_o = vbt.Portfolio.from_signals(close_oos, entries_global[p_tuple].iloc[split_idx:], exits_global[p_tuple].iloc[split_idx:], freq='D')
        oos_rets_list.append(pf_o.returns())
        oos_trd.append(pf_o.trades.count())
        oos_win.append((pf_o.trades.pnl > 0).sum())
        
    blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
    blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
    avg_trd_is = np.mean(is_trd)
    win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
    avg_trd_oos = np.mean(oos_trd)
    win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
    del entries_global, exits_global, ema_df
    gc.collect()

    # --- 🔥 INSERITI METADATI RAW PER EXPORT COMPLETO ---
    return {
        "Parametri": param_str,
        "Parametri_Raw": [[int(p[0]), int(p[1]), int(p[2])] for p in top_stable_params],
        "Giorni_Anno": giorni_anno,
        "Inizio_OOS": data_inizio_oos,
        
        # Strategy Metrics
        "SHP_IS": shp_is, "SHP_OOS": shp_oos,
        "CAGR_IS": cagr_is, "CAGR_OOS": cagr_oos,
        "DD_IS": dd_is, "DD_OOS": dd_oos,
        "VOL_IS": vol_is, "VOL_OOS": vol_oos,
        "SORT_IS": sort_is, "SORT_OOS": sort_oos,
        "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        
        # Benchmark Metrics (Buy & Hold)
        "BH_SHP_IS": shp_bh_is, "BH_SHP_OOS": shp_bh_oos,
        "BH_CAGR_IS": cagr_bh_is, "BH_CAGR_OOS": cagr_bh_oos,
        "BH_DD_IS": dd_bh_is, "BH_DD_OOS": dd_bh_oos,
        "BH_VOL_IS": vol_bh_is, "BH_VOL_OOS": vol_bh_oos,
        "BH_SORT_IS": sort_bh_is, "BH_SORT_OOS": sort_bh_oos,
        
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_ensemble(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trouvata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE SPLIT 60/40 ENSEMBLE (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Ensemble)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

# ==============================================================================
# 🔥 6. SALVATAGGIO CONFIGURAZIONE JSON PER INTERFACCIA ENSEMBLE
# ==============================================================================
if len(report_finale) > 0:
    export_dir = "progetto_trading_data"
    os.makedirs(export_dir, exist_ok=True)
    path_export = os.path.join(export_dir, "config_3ema.json")
    
    config_json = {
        "NOME_STRATEGIA": "3EMA_3D_Convolution_Ensemble",
        "DATA_GENERAZIONE": "2026-06-21",
        "ASSETS": {}
    }
    
    for item in report_finale:
        ticker = item["Ticker"]
        config_json["ASSETS"][ticker] = {
            "Cluster": item["Cluster"],
            "Data_Inizio_OOS": item["Inizio_OOS"],
            "Giorni_Anno": int(item["Giorni_Anno"]),
            "Numero_Modelli_Ensemble": len(item["Parametri_Raw"]),
            "Parametri_Ottimi": item["Parametri_Raw"],
            "Alpha_OOS": float(item["Alpha_OOS"]),
            
            "Metrics_In_Sample": {
                "Sharpe_IS": float(item["SHP_IS"]),
                "Sortino_IS": float(item["SORT_IS"]),
                "CAGR_IS": float(item["CAGR_IS"]),
                "Volatilità_IS": float(item["VOL_IS"]),
                "MaxDD_IS": float(item["DD_IS"]),
                "Trades_IS": float(item["TRD_IS"]),
                "WinRate_IS": float(item["WIN_IS"])
            },
            "Metrics_Out_Of_Sample": {
                "Sharpe_OOS": float(item["SHP_OOS"]),
                "Sortino_OOS": float(item["SORT_OOS"]),
                "CAGR_OOS": float(item["CAGR_OOS"]),
                "Volatilità_OOS": float(item["VOL_OOS"]),
                "MaxDD_OOS": float(item["DD_OOS"]),
                "Trades_OOS": float(item["TRD_OOS"]),
                "WinRate_OOS": float(item["WIN_OOS"])
            },
            "Benchmark_In_Sample": {
                "Sharpe_BH_IS": float(item["BH_SHP_IS"]),
                "Sortino_BH_IS": float(item["BH_SORT_IS"]),
                "CAGR_BH_IS": float(item["BH_CAGR_IS"]),
                "Volatilità_BH_IS": float(item["BH_VOL_IS"]),
                "MaxDD_BH_IS": float(item["BH_DD_IS"])
            },
            "Benchmark_Out_Of_Sample": {
                "Sharpe_BH_OOS": float(item["BH_SHP_OOS"]),
                "Sortino_BH_OOS": float(item["BH_SORT_OOS"]),
                "CAGR_BH_OOS": float(item["BH_CAGR_OOS"]),
                "Volatilità_BH_OOS": float(item["BH_VOL_OOS"]),
                "MaxDD_BH_OOS": float(item["BH_DD_OOS"])
            }
        }
        
    with open(path_export, "w") as f:
        json.dump(config_json, f, indent=4)
        
    print(f"\n💾 Configurazione globale salvata con successo in: {path_export} 🚀")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore Split 60/40 Asincrono (Convoluzione 3D + Ensemble) su 90 asset indipendenti.
📦 Griglia tridimensionale generata: 10320 combinazioni valide.

⌛ ELABORAZIONE COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: [(10,36,245), (18,20,220), (8,36,100)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.44 | Cagr: 62.7% | DD: -54.9% | Sort: 1.46 | Trd: 6.0 | Win: 50.0%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: -0.31 | Cagr: -12.0% | DD: -31.6% | Sort: -0.28 | Trd: 5.7 | Win: 23.5%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  41.98%

⌛ ELABORAZIONE COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: [(4,84,135), (4,84,155), (6,76,145)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.47 | Cagr: 44.3% | DD: -18.5% | Sort: 1.86 | Trd: 5.3 | Win: 56.2%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2